In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('/content/Agrofood_co2_emission_cleaned.csv')

In [ ]:
df.head()

,Area,Year,Savanna fires,Forest fires,Crop Residues,Rice Cultivation,Drained organic soils (CO2),Pesticides Manufacturing,Food Transport,Forestland,...,Manure Management,Fires in organic soils,Fires in humid tropical forests,On-farm energy use,Rural population,Urban population,Total Population - Male,Total Population - Female,total_emission,Average Temperature °C
0,Afghanistan,1990,14.7237,0.0557,205.6077,686.00,0.0,11.807483,63.1152,-2388.803,...,319.1763,0.0,0.0,135.5461,9655167.0,2593947.0,5348387.0,5346409.0,2198.963539,0.536167
1,Afghanistan,1991,14.7237,0.0557,209.4971,678.16,0.0,11.712073,61.2125,-2388.803,...,342.3079,0.0,0.0,135.5461,10230490.0,2763167.0,5372959.0,5372208.0,2323.876629,0.020667
2,Afghanistan,1992,14.7237,0.0557,196.5341,686.00,0.0,11.712073,53.3170,-2388.803,...,349.1224,0.0,0.0,135.5461,10995568.0,2985663.0,6028494.0,6028939.0,2356.304229,-0.259583
3,Afghanistan,1993,14.7237,0.0557,230.8175,686.00,0.0,11.712073,54.3617,-2388.803,...,352.2947,0.0,0.0,135.5461,11858090.0,3237009.0,7003641.0,7000119.0,2368.470529,0.101917
4,Afghanistan,1994,14.7237,0.0557,242.0494,705.60,0.0,11.712073,53.9874,-2388.803,...,367.6784,0.0,0.0,135.5461,12690115.0,3482604.0,7733458.0,7722096.0,2500.768729,0.372250


Total emissions alone are misleading without population context.

In [ ]:
df['total_population'] = (
    df['Total Population - Male'] + df['Total Population - Female']
)

df['emission_per_capita'] = df['total_emission'] / df['total_population']


Instead of raw urban & rural population:
Ratios generalize better than absolute numbers
Strong correlation with emissions (as you already showed for China)

In [ ]:
df['urban_ratio'] = df['Urban population'] / df['total_population']
df['rural_ratio'] = df['Rural population'] / df['total_population']


Forestland and deforestation effects matter more relative to emissions.

In [ ]:
df['forest_emission_ratio'] = df['Forestland'] / df['total_emission']
df['net_forest_change_abs'] = df['Net Forest conversion'].abs()


Instead of 20+ correlated emission columns, group them logically.

In [ ]:
df['fire_emissions'] = (
    df['Savanna fires'] +
    df['Forest fires'] +
    df['Fires in organic soils'] +
    df['Fires in humid tropical forests']
)


In [ ]:
df['agriculture_emissions'] = (
    df['Rice Cultivation'] +
    df['Crop Residues'] +
    df['Manure applied to Soils'] +
    df['Manure left on Pasture'] +
    df['Manure Management']
)


In [ ]:
df['industrial_emissions'] = (
    df['Fertilizers Manufacturing'] +
    df['Pesticides Manufacturing'] +
    df['Food Processing'] +
    df['IPPU']
)


In [ ]:
df['energy_intensity'] = (
    df['On-farm energy use'] + df['On-farm Electricity Use']
)


In [ ]:
df.drop(columns=['Area'], inplace=True)

In [ ]:
corr = df.corr()['total_emission'].abs().sort_values(ascending=False)
corr.head(15)


,total_emission
total_emission,1.000000
Urban population,0.907774
Agrifood Systems Waste Disposal,0.879840
Food Household Consumption,0.861939
industrial_emissions,0.856643
agriculture_emissions,0.854875
IPPU,0.851667
Manure applied to Soils,0.845958
Food Packaging,0.842555
Crop Residues,0.836294


In [ ]:
X = df.drop(columns=['total_emission'])
y = df['total_emission']

corr = X.corrwith(y).abs()
selected_corr_features = corr[corr > 0.05].index
X_corr = X[selected_corr_features]


In [ ]:
corr_matrix = X_corr.corr().abs()
upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

high_corr_features = [
    col for col in upper.columns if any(upper[col] > 0.9)
]

X_clean = X_corr.drop(columns=high_corr_features)


In [ ]:
from sklearn.feature_selection import SelectKBest, f_regression

selector = SelectKBest(score_func=f_regression, k=15)
X_selected = selector.fit_transform(X_clean, y)

selected_features = X_clean.columns[selector.get_support()]
print("Final Selected Features:\n", selected_features)


Final Selected Features:
 Index(['Savanna fires', 'Forest fires', 'Crop Residues', 'Rice Cultivation',
       'Drained organic soils (CO2)', 'Pesticides Manufacturing',
       'Food Transport', 'Forestland', 'Net Forest conversion',
       'Food Household Consumption', 'Food Retail', 'On-farm Electricity Use',
       'Manure left on Pasture', 'Fires in organic soils'],
      dtype='object')


/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:783: UserWarning: k=15 is greater than n_features=14. All the features will be returned.
  warnings.warn(


In [ ]:
from sklearn.model_selection import train_test_split

X_final = X_clean[selected_features]

X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, test_size=0.2, random_state=42
)


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
X_train_scaled = pd.DataFrame(
    X_train_scaled, columns=X_train.columns, index=X_train.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled, columns=X_test.columns, index=X_test.index
)


In [ ]:
# Save features
X_train_scaled.to_csv('X_train_scaled.csv', index=False)
X_test_scaled.to_csv('X_test_scaled.csv', index=False)

# Save targets
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)


In [ ]:
import joblib
joblib.dump(scaler, 'scaler.pkl')

['scaler.pkl']